# 🚀 Bài Tập Khoa Học Dữ Liệu - Pipeline Hoàn Chỉnh

**1. Chuẩn hóa | 2. Tiền xử lý | 3. Model | 4. Đánh giá | 5. Giải thích**

---

## Bước 0: Import thư viện và tải dữ liệu


In [ ]:
# Chạy 1 lần nếu chưa cài
# !pip install pandas numpy scikit-learn matplotlib seaborn pyodbc sqlalchemy scipy pandasql
print("✅ Sẵn sàng!")


✅ Sẵn sàng!


## Bước 1: Import thư viện 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('results/figures', exist_ok=True)
os.makedirs('data', exist_ok=True)

from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import NearestNeighbors

print("✅ Import thư viện thành công!")


## Bước 2: Kết nối SQL & Load data

In [ ]:
import pyodbc

def get_dataframe(query="SELECT * FROM Food_Delivery_Times"):
    SERVER   = "DESKTOP-D1HC55M"
    DATABASE = "abc"
    conn_str = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        "Trusted_Connection=yes;"
    )
    conn = pyodbc.connect(conn_str)
    df   = pd.read_sql(query, conn)
    conn.close()
    return df

df = get_dataframe()

# Ép kiểu ngay sau khi load
df['Delivery_Time_min']      = pd.to_numeric(df['Delivery_Time_min'],      errors='coerce')
df['Distance_km']            = pd.to_numeric(df['Distance_km'],            errors='coerce')
df['Preparation_Time_min']   = pd.to_numeric(df['Preparation_Time_min'],   errors='coerce')
df['Courier_Experience_yrs'] = pd.to_numeric(df['Courier_Experience_yrs'], errors='coerce')

print("✅ Kết nối SQL Server thành công!")
print(f"Shape: {df.shape}")
df.head()  # hiện bảng đẹp trong notebook


## Bước 3: Thông tin dữ liệu

In [ ]:
print("="*50)
print("THÔNG TIN DỮ LIỆU")
print("="*50)
print(f"Số dòng  : {df.shape[0]}")
print(f"Số cột   : {df.shape[1]}")
print(f"\nKiểu dữ liệu:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df.describe()


## Bước 4: SQL Queries phân tích

In [ ]:
import pandasql as psql

queries = {
    "Q1 - Thời tiết ảnh hưởng thế nào": """
        SELECT Weather, COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut,
               ROUND(MIN(Delivery_Time_min),2) AS min_phut,
               ROUND(MAX(Delivery_Time_min),2) AS max_phut
        FROM df GROUP BY Weather ORDER BY tb_phut DESC
    """,
    "Q2 - Mức độ tắc đường": """
        SELECT Traffic_Level, COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut
        FROM df GROUP BY Traffic_Level ORDER BY tb_phut DESC
    """,
    "Q3 - Loại phương tiện": """
        SELECT Vehicle_Type, COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut
        FROM df GROUP BY Vehicle_Type ORDER BY tb_phut
    """,
    "Q4 - Khung giờ trong ngày": """
        SELECT Time_of_Day, COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut
        FROM df GROUP BY Time_of_Day ORDER BY tb_phut DESC
    """,
    "Q5 - Kinh nghiệm tài xế": """
        SELECT CASE
                   WHEN Courier_Experience_yrs < 1 THEN 'Mới (<1 năm)'
                   WHEN Courier_Experience_yrs < 3 THEN 'Trung bình (1-3 năm)'
                   ELSE 'Có kinh nghiệm (>3 năm)'
               END AS nhom_kn,
               COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut
        FROM df
        GROUP BY CASE
                     WHEN Courier_Experience_yrs < 1 THEN 'Mới (<1 năm)'
                     WHEN Courier_Experience_yrs < 3 THEN 'Trung bình (1-3 năm)'
                     ELSE 'Có kinh nghiệm (>3 năm)'
                 END
        ORDER BY tb_phut DESC
    """,
    "Q6 - Top 10 đơn giao lâu nhất": """
        SELECT * FROM df
        WHERE Delivery_Time_min > 60
        ORDER BY Delivery_Time_min DESC
        LIMIT 10
    """,
    "Q7 - Tắc đường + Thời tiết kết hợp": """
        SELECT Traffic_Level, Weather,
               COUNT(*) AS tong_don,
               ROUND(AVG(Delivery_Time_min),2) AS tb_phut
        FROM df GROUP BY Traffic_Level, Weather
        ORDER BY tb_phut DESC LIMIT 10
    """
}

for title, q in queries.items():
    print(f"\n{'='*50}")
    print(f"  {title}")
    print('='*50)
    try:
        result = psql.sqldf(q, locals())
        display(result)          # hiện bảng đẹp trong notebook
    except Exception as e:
        print(f"Lỗi: {e}")


## Bước 5: Biểu đồ EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Phân tích Dữ liệu Giao Đồ Ăn', fontsize=16, fontweight='bold')

# 1. Histogram
axes[0,0].hist(df['Delivery_Time_min'].dropna(), bins=30,
               color='steelblue', edgecolor='white')
axes[0,0].set_title('Phân phối Thời gian Giao Hàng')
axes[0,0].set_xlabel('Thời gian (phút)'); axes[0,0].set_ylabel('Số đơn')

# 2. Traffic vs time
df.groupby(df['Traffic_Level'].astype(str))['Delivery_Time_min']\
  .mean().sort_values()\
  .plot(kind='bar', ax=axes[0,1], color='orange', edgecolor='black')
axes[0,1].set_title('Tắc Đường vs Thời gian Giao')
axes[0,1].tick_params(axis='x', rotation=30)

# 3. Weather vs time
df.groupby(df['Weather'].astype(str))['Delivery_Time_min']\
  .mean().sort_values()\
  .plot(kind='bar', ax=axes[0,2], color='green', edgecolor='black')
axes[0,2].set_title('Thời Tiết vs Thời gian Giao')
axes[0,2].tick_params(axis='x', rotation=30)

# 4. Scatter
color_list = ['green','orange','red','blue','purple']
color_map  = {v: color_list[i % len(color_list)]
              for i, v in enumerate(sorted(df['Traffic_Level'].astype(str).unique()))}
for tval, grp in df.groupby(df['Traffic_Level'].astype(str)):
    axes[1,0].scatter(pd.to_numeric(grp['Distance_km'], errors='coerce'),
                      pd.to_numeric(grp['Delivery_Time_min'], errors='coerce'),
                      label=tval, alpha=0.4, s=10, color=color_map.get(tval,'blue'))
axes[1,0].set_title('Khoảng cách vs Thời gian Giao')
axes[1,0].legend(title='Tắc đường', markerscale=3)

# 5. Boxplot
vehicle_types = sorted(df['Vehicle_Type'].astype(str).unique())
vehicle_data  = [pd.to_numeric(df[df['Vehicle_Type'].astype(str)==v]['Delivery_Time_min'],
                               errors='coerce').dropna().values
                 for v in vehicle_types]
axes[1,1].boxplot(vehicle_data, labels=vehicle_types)
axes[1,1].set_title('Phương Tiện vs Thời gian Giao')
axes[1,1].tick_params(axis='x', rotation=15)

# 6. Heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=axes[1,2])
axes[1,2].set_title('Ma trận tương quan')

plt.tight_layout()
plt.savefig('results/figures/EDA_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA hoàn thành!")


## Bước 6: Tiền xử lý

In [ ]:
df_model = df.copy()
if 'Order_ID' in df_model.columns:
    df_model.drop(columns=['Order_ID'], inplace=True)

# Ép kiểu
num_cols_raw = ['Distance_km','Preparation_Time_min',
                'Courier_Experience_yrs','Delivery_Time_min']
cat_cols_raw = ['Weather','Traffic_Level','Time_of_Day','Vehicle_Type']

for col in num_cols_raw:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')
for col in cat_cols_raw:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(str).str.strip()

# Fill missing
for col in df_model.select_dtypes(include=[np.number]).columns:
    df_model[col].fillna(df_model[col].median(), inplace=True)
for col in df_model.select_dtypes(include='object').columns:
    df_model[col].fillna(df_model[col].mode()[0], inplace=True)

# Encode
for col in df_model.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Tách X, y
TARGET   = 'Delivery_Time_min'
X        = df_model.drop(columns=[TARGET]).values.astype(np.float64)
y        = df_model[TARGET].values.astype(np.float64)

# Xóa NaN còn sót
if np.isnan(X).sum() > 0 or np.isnan(y).sum() > 0:
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X, y = X[mask], y[mask]

# Scale & split
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

np.save('data/X_train.npy', X_train); np.save('data/X_test.npy',  X_test)
np.save('data/y_train.npy', y_train); np.save('data/y_test.npy',  y_test)

print(f"✅ Tiền xử lý hoàn thành!")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"NaN trong X_train: {np.isnan(X_train).sum()}")


## Bước 7: Linear & Polynomial Regression

In [ ]:
def danh_gia(y_true, y_pred, ten=""):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  [{ten}]  RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}")
    return rmse, mae, r2

results = []
feature_names = df_model.drop(columns=[TARGET]).columns.tolist()

# Linear
print("="*50)
print("LINEAR REGRESSION")
print("="*50)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr      = lr.predict(X_test)
rmse, mae, r2  = danh_gia(y_test, y_pred_lr, "Linear")
results.append({'Mo_hinh':'Linear','Bac':1,'RMSE':rmse,'MAE':mae,'R2':r2})

print("\nHệ số hồi quy:")
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': lr.coef_})
coef_df['|Coefficient|'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('|Coefficient|', ascending=False)
display(coef_df)

# Polynomial
print("\n" + "="*50)
print("POLYNOMIAL REGRESSION")
print("="*50)
for deg in [2, 3, 4]:
    poly       = PolynomialFeatures(degree=deg, include_bias=False)
    X_tr_poly  = poly.fit_transform(X_train)
    X_te_poly  = poly.transform(X_test)
    m_poly     = LinearRegression()
    m_poly.fit(X_tr_poly, y_train)
    y_pred_p   = m_poly.predict(X_te_poly)
    rmse, mae, r2 = danh_gia(y_test, y_pred_p, f"Polynomial deg={deg}")
    results.append({'Mo_hinh':f'Polynomial','Bac':deg,'RMSE':rmse,'MAE':mae,'R2':r2})

df_results = pd.DataFrame(results)
df_results.to_csv('results/model_comparison.csv', index=False)

print("\n📊 Bảng so sánh:")
display(df_results)

best = df_results.loc[df_results['RMSE'].idxmin()]
print(f"\n🏆 Tốt nhất: {best['Mo_hinh']} bậc {int(best['Bac'])} — RMSE={best['RMSE']:.4f}")


## Bước 8: Biểu đồ so sánh mô hình

In [ ]:
labels = [f"{r['Mo_hinh']}\n(bậc {int(r['Bac'])})"
          for _, r in df_results.iterrows()]
colors = ['steelblue','orange','green','red']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(labels, df_results['RMSE'], color=colors)
axes[0].set_title('So sánh RMSE (càng thấp càng tốt)')
axes[0].set_ylabel('RMSE (phút)'); axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_results['RMSE']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(labels, df_results['R2'], color=colors)
axes[1].set_title('So sánh R² (càng cao càng tốt)')
axes[1].set_ylabel('R²'); axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_results['R2']):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

axes[2].scatter(y_test, y_pred_lr, alpha=0.3, s=10, color='steelblue')
axes[2].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', label='Lý tưởng')
axes[2].set_title('Thực tế vs Dự đoán (Linear)')
axes[2].set_xlabel('Thực tế (phút)'); axes[2].set_ylabel('Dự đoán (phút)')
axes[2].legend()

plt.tight_layout()
plt.savefig('results/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Biểu đồ mô hình hoàn thành!")


## Bước 9: DBSCAN

In [ ]:
print("="*50)
print("DBSCAN — TỰ ĐỘNG TÌM eps")
print("="*50)

# K-distance graph
k    = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X_train)
distances, _ = nbrs.kneighbors(X_train)
k_distances  = np.sort(distances[:, k-1])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
eps_90 = np.percentile(k_distances, 90)
plt.axhline(y=eps_90, color='red', linestyle='--',
            label=f'90th percentile = {eps_90:.3f}')
plt.title('K-Distance Graph'); plt.xlabel('Điểm'); plt.ylabel('Khoảng cách')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig('results/figures/kdistance.png', dpi=150)
plt.show()

# Thử tham số
best_eps, best_min_samp = float(eps_90), 5
print("\nThử nghiệm tham số:")
for eps_val in [eps_90*0.5, eps_90*0.7, eps_90, eps_90*1.5, eps_90*2.0]:
    for min_s in [3, 5, 10]:
        lbl = DBSCAN(eps=eps_val, min_samples=min_s).fit_predict(X_train)
        n_out = np.sum(lbl == -1)
        pct   = n_out / len(lbl) * 100
        tag   = " ✅ (5-15%)" if 5 <= pct <= 15 else ""
        print(f"  eps={eps_val:.3f}, min_s={min_s} → {n_out} ngoại lệ ({pct:.1f}%){tag}")
        if 5 <= pct <= 15:
            best_eps, best_min_samp = eps_val, min_s

# Chạy DBSCAN tốt nhất
labels_db  = DBSCAN(eps=best_eps, min_samples=best_min_samp).fit_predict(X_train)
n_out      = np.sum(labels_db == -1)
n_total    = len(labels_db)
print(f"\n✅ eps={best_eps:.4f}, min_samples={best_min_samp}")
print(f"Ngoại lệ: {n_out}/{n_total} ({n_out/n_total*100:.1f}%)")

mask = labels_db != -1 if np.sum(labels_db != -1) >= 10 else np.ones(len(labels_db), dtype=bool)
X_train_clean = X_train[mask]
y_train_clean = y_train[mask]
print(f"Dữ liệu sau DBSCAN: {X_train_clean.shape[0]} điểm")


## Bước 10: So sánh trước/sau DBSCAN

In [ ]:
results_dbscan = []
print("Kết quả sau DBSCAN:")
for deg_d in [1, 2, 3, 4]:
    if deg_d == 1:
        m = LinearRegression()
        m.fit(X_train_clean, y_train_clean)
        y_p = m.predict(X_test)
        ten = "Linear (DBSCAN)"
    else:
        poly_d = PolynomialFeatures(degree=deg_d, include_bias=False)
        m = LinearRegression()
        m.fit(poly_d.fit_transform(X_train_clean), y_train_clean)
        y_p = m.predict(poly_d.transform(X_test))
        ten = f"Poly deg={deg_d} (DBSCAN)"
    rmse, mae, r2 = danh_gia(y_test, y_p, ten)
    results_dbscan.append({'Mo_hinh': ten, 'Bac': deg_d,
                           'RMSE': rmse, 'MAE': mae, 'R2': r2})

df_dbscan = pd.DataFrame(results_dbscan)

# Biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)
sc    = axes[0].scatter(X_pca[:,0], X_pca[:,1],
                        c=labels_db, cmap='viridis', s=5, alpha=0.5)
axes[0].set_title('DBSCAN Clusters (PCA 2D)')
plt.colorbar(sc, ax=axes[0], label='Cluster (-1=Outlier)')

x_pos = np.arange(4); w = 0.35
axes[1].bar(x_pos-w/2, df_results['RMSE'],  w, label='Trước DBSCAN', color='steelblue')
axes[1].bar(x_pos+w/2, df_dbscan['RMSE'],   w, label='Sau DBSCAN',   color='tomato')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['Linear','Poly 2','Poly 3','Poly 4'], rotation=15)
axes[1].set_title('RMSE Trước vs Sau DBSCAN')
axes[1].set_ylabel('RMSE'); axes[1].legend()
for i in range(4):
    axes[1].text(i-w/2, df_results['RMSE'].iloc[i]+0.01,
                 f"{df_results['RMSE'].iloc[i]:.3f}", ha='center', fontsize=8)
    axes[1].text(i+w/2, df_dbscan['RMSE'].iloc[i]+0.01,
                 f"{df_dbscan['RMSE'].iloc[i]:.3f}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/dbscan_results.png', dpi=150, bbox_inches='tight')
plt.show()
display(df_dbscan)
print("✅ DBSCAN hoàn thành!")


## Bước 11: Tổng kết

In [ ]:
print("="*65)
print("         TỔNG KẾT KẾT QUẢ TOÀN BỘ DỰ ÁN")
print("="*65)

print(f"\n📌 Dữ liệu: {len(df)} dòng | {X_train.shape[1]} features")
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

print("\n📌 Mô hình gốc:")
display(df_results)

print("\n📌 Sau DBSCAN:")
display(df_dbscan)

best_all = df_dbscan.loc[df_dbscan['RMSE'].idxmin()]
print(f"\n🏆 Mô hình tốt nhất: {best_all['Mo_hinh']}")
print(f"   RMSE = {best_all['RMSE']:.4f} phút")
print(f"   R²   = {best_all['R2']:.4f} ({best_all['R2']*100:.1f}% sự biến động)")
print("\n✅ SẴN SÀNG BÁO CÁO!")
